# 01 · Simulate the loan book

**The loans are SIMULATED. The interest rates and province mix are REAL** (Bank of Canada,
Canadian Bankers Association, Statistics Canada).

No Canadian lender publishes customer-level loan data, so we generate a realistic book of
50,000 loans and price every loan off the real rate in the month it was signed. Banks do the
same thing when they build and test risk models.

**What this notebook produces:** `data/loans.csv` — one row per loan, including the
renewal rate and payment shock for fixed mortgages renewing in 2025–2027.

**Key idea:** a Canadian mortgage is paid off over 25 years, but the rate is only locked for
5. A family that signed at ~1.9% in 2021 must renew in 2026 at whatever the rate is then.
The jump in their monthly payment is the **payment shock**.

In [1]:
# Make file paths work whether the notebook runs from the project folder or from notebooks/
import os
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

import numpy as np
import pandas as pd
import yaml

from src.logger import get_logger

log = get_logger("01_simulate_loans")

## 1. Load the settings and the real rates
Every number used below (product mix, loan sizes, spreads...) comes from `config/pipeline.yml`,
where each one is tagged `[brief]` (researched) or `[ASSUMPTION]` (a judgement call).

In [2]:
config = yaml.safe_load(open("config/pipeline.yml"))
portfolio = config["portfolio"]
risk = config["risk"]
window = config["window"]

rng = np.random.default_rng(config["seed"])  # fixed seed -> the same 50,000 loans every run
n = portfolio["n_loans"]

rates = pd.read_csv("data/rates.csv", parse_dates=["month"])
first_month = rates["month"].min()
last_month_num = len(rates) - 1  # one row per month, so row number = month number

# Month start dates, long enough to name renewals up to 10 years past the data
calendar = pd.date_range(first_month, periods=len(rates) + 120, freq="MS")


def to_month_num(date_text):
    """Months since the first month of rate data (Jan 2015 = 0)."""
    date = pd.Timestamp(date_text)
    return (date.year - first_month.year) * 12 + (date.month - first_month.month)


log.info(f"{n:,} loans to simulate; real rates cover {len(rates)} months")
rates.tail(3)

2026-10-17 19:01:59,258 | 01_simulate_loans | INFO | 50,000 loans to simulate; real rates cover 141 months


,month,policy_rate,prime_rate,goc_5yr_yield,posted_5yr_mortgage,source
138,2026-07-01,2.25,4.45,3.163182,6.09,Bank of Canada Valet API (real)
139,2026-08-01,2.25,4.45,3.281500,6.09,Bank of Canada Valet API (real)
140,2026-09-01,2.25,4.45,3.568000,6.09,Bank of Canada Valet API (real)


## 2. Where the borrowers live — from REAL data
Instead of guessing province shares, we use where Canada's mortgages actually are:
the Canadian Bankers Association's count of mortgages by region (latest month). The CBA
reports the four Atlantic provinces together, so we split Atlantic Canada by each province's
real population (Statistics Canada). Needs `python src\fetch_provincial_data.py` first.

In [3]:
arrears = pd.read_csv("data/mortgage_arrears.csv", parse_dates=["month"])
unemployment = pd.read_csv("data/unemployment.csv", parse_dates=["month"])
province_region = config["sources"]["cba_arrears"]["province_region"]

# Share of Canada's mortgages in each region, latest month
latest = arrears[(arrears["month"] == arrears["month"].max()) & (arrears["region"] != "Canada")]
region_share = latest.set_index("region")["total_mortgages"] / latest["total_mortgages"].sum()

# Each province's share of its region's population (only matters for Atlantic Canada)
latest_pop = unemployment[unemployment["month"] == unemployment["month"].max()]
population = latest_pop.set_index("province")["population_thousands"]

provinces = pd.DataFrame({"region": pd.Series(province_region)})
provinces["population_thousands"] = population
provinces["share_of_region"] = provinces["population_thousands"] / provinces.groupby("region")["population_thousands"].transform("sum")
provinces["share_of_mortgages"] = provinces["region"].map(region_share) * provinces["share_of_region"]
shares = provinces["share_of_mortgages"].to_dict()

print(f"Shares sum to {sum(shares.values()):.6f}; based on CBA data for {latest['month'].max():%b %Y}")
provinces.sort_values("share_of_mortgages", ascending=False).round(4)

Shares sum to 1.000000; based on CBA data for Jul 2026


,region,population_thousands,share_of_region,share_of_mortgages
ON,Ontario,13798.4,1.0000,0.4378
QC,Quebec,7577.2,1.0000,0.1903
BC,British Columbia,4844.2,1.0000,0.1400
AB,Alberta,4164.0,1.0000,0.1158
NS,Atlantic,927.1,0.4048,0.0276
SK,Saskatchewan,988.8,1.0000,0.0243
MB,Manitoba,1191.8,1.0000,0.0233
NB,Atlantic,731.3,0.3193,0.0218
NL,Atlantic,475.6,0.2076,0.0142
PE,Atlantic,156.4,0.0683,0.0047


## 3. Who the borrowers are
Product, credit score, fixed vs variable, loan size and signing month are drawn at random
using the shares in the config; province uses the real shares above.

In [4]:
loans = pd.DataFrame({"loan_id": np.arange(1, n + 1)})

mix = portfolio["product_mix"]
loans["product"] = rng.choice(list(mix), size=n, p=list(mix.values()))

loans["province"] = rng.choice(list(shares), size=n, p=list(shares.values()))

score = portfolio["credit_score"]
loans["credit_score"] = rng.normal(score["mean"], score["sd"], n).round().clip(300, 900).astype(int)

# Fixed or variable: mortgages are a mix, HELOCs always float with prime
is_mortgage = loans["product"] == "mortgage"
loans["rate_type"] = "fixed"
loans.loc[loans["product"] == "heloc", "rate_type"] = "variable"
fixed_share = portfolio["mortgage_fixed_share"]
loans.loc[is_mortgage, "rate_type"] = rng.choice(
    ["fixed", "variable"], size=is_mortgage.sum(), p=[fixed_share, 1 - fixed_share]
)

# Loan size: uniform between each product's min and max
loans["original_balance"] = 0.0
for product, (low, high) in portfolio["original_balance"].items():
    is_product = loans["product"] == product
    loans.loc[is_product, "original_balance"] = rng.uniform(low, high, is_product.sum()).round(2)

# Signing month: any month from origination_start to the end of the data.
# Auto loans are paid off after 72 months, so they must start late enough to still be open
# when the performance window begins.
earliest = to_month_num(window["origination_start"])
latest = to_month_num(window["performance_end"])
loans["orig_month_num"] = rng.integers(earliest, latest + 1, n)
is_auto = loans["product"] == "auto"
auto_earliest = to_month_num(window["performance_start"]) - risk["auto"]["term_months"] + 1
loans.loc[is_auto, "orig_month_num"] = rng.integers(auto_earliest, latest + 1, is_auto.sum())
loans["origination_month"] = calendar[loans["orig_month_num"]]

loans.head()

,loan_id,product,province,credit_score,rate_type,original_balance,orig_month_num,origination_month
0,1,card,BC,795,fixed,3609.60,75,2021-04-01
1,2,heloc,ON,629,variable,205171.32,54,2019-07-01
2,3,card,ON,759,fixed,6661.09,107,2023-12-01
3,4,card,QC,721,fixed,1976.81,128,2025-09-01
4,5,mortgage,ON,886,variable,881433.51,9,2015-10-01


## 4. Interest rate at signing, from the real rates that month
- **Fixed mortgage** = 5-year Government of Canada bond yield + 1.40pp. (The Bank of Canada's
  "posted" rate is a sticker price banks discount heavily from — using it would overstate rates.)
- **Variable mortgage / HELOC** = prime rate +/- a spread.
- **Better credit score → lower rate**, but how much depends on the product: mortgages barely
  price on score, credit cards a lot.

In [5]:
goc_5yr = rates["goc_5yr_yield"].to_numpy()  # look up by month number
prime = rates["prime_rate"].to_numpy()
goc_at_signing = goc_5yr[loans["orig_month_num"]]
prime_at_signing = prime[loans["orig_month_num"]]

# +1 for every 100 points above the reference score
score_vs_reference = (loans["credit_score"] - score["reference"]) / 100

m, h, a, c = risk["mortgage"], risk["heloc"], risk["auto"], risk["card"]
fixed_mortgage = np.maximum(
    goc_at_signing + m["fixed_spread_over_goc_5yr"] - m["score_sensitivity_per_100pts"] * score_vs_reference,
    m["fixed_rate_floor"],
)
variable_mortgage = np.maximum(
    prime_at_signing + m["variable_spread_over_prime"] - m["score_sensitivity_per_100pts"] * score_vs_reference,
    m["variable_rate_floor"],
)
heloc = prime_at_signing + h["spread_over_prime"] - h["score_sensitivity_per_100pts"] * score_vs_reference
auto = goc_at_signing + a["spread_over_goc_5yr"] - a["score_sensitivity_per_100pts"] * score_vs_reference
card = np.clip(c["base_apr"] - c["score_sensitivity_per_100pts"] * score_vs_reference, *c["apr_range"])

is_fixed_mortgage = is_mortgage & (loans["rate_type"] == "fixed")
is_variable_mortgage = is_mortgage & (loans["rate_type"] == "variable")
loans["rate_at_signing"] = np.select(
    [is_fixed_mortgage, is_variable_mortgage, loans["product"] == "heloc", is_auto, loans["product"] == "card"],
    [fixed_mortgage, variable_mortgage, heloc, auto, card],
).round(2)

loans.groupby("product")["rate_at_signing"].describe().round(2)

,count,mean,std,min,25%,50%,75%,max
product,,,,,,,,
auto,10015.0,5.43,1.48,0.88,4.38,5.47,6.50,10.70
card,17613.0,18.92,3.80,9.99,16.30,19.00,21.53,29.99
heloc,4916.0,4.47,1.57,2.45,3.15,4.06,5.34,8.32
mortgage,17456.0,3.24,1.24,1.15,2.18,2.98,4.31,6.41


## 5. Fixed-mortgage renewals in 2025–2027, and the payment shock
Only **fixed** mortgages get a renewal shock — variable rates already move with prime every month.

- **Old rate:** locked in at the start of the current 5-year term.
- **New rate:** the real rate in the renewal month. For renewals after Sep 2026 (the last real
  month) we hold the last real rate flat — **no forecasting**, labelled in `new_rate_source`.
- **Payment shock %** = new monthly payment ÷ old monthly payment − 1.

In [6]:
def monthly_payment(balance, annual_rate_pct, months):
    """Standard mortgage payment formula."""
    r = annual_rate_pct / 100 / 12
    return balance * r / (1 - (1 + r) ** -months)


def balance_after(balance, annual_rate_pct, payment, months_paid):
    """What is still owed after `months_paid` payments."""
    r = annual_rate_pct / 100 / 12
    growth = (1 + r) ** months_paid
    return balance * growth - payment * (growth - 1) / r


term = m["term_months"]
amortization = m["amortization_months"]
window_start = to_month_num(window["renewal_window_start"])
window_end = to_month_num(window["renewal_window_end"])
orig = loans["orig_month_num"].to_numpy()

# First renewal on or after the window start. A loan signed in 2016 renewed in 2021 and
# renews again in 2026 - the 2026 one is the renewal we analyse.
terms_completed = np.maximum(np.ceil((window_start - orig) / term), 1).astype(int)
renewal_num = orig + term * terms_completed
term_start_num = renewal_num - term
renews = (is_fixed_mortgage & (renewal_num <= window_end)).to_numpy()

renewal_rate_month = np.minimum(renewal_num, last_month_num)  # future renewals: last real rate, held flat
mortgage_score_adj = m["score_sensitivity_per_100pts"] * score_vs_reference
old_rate = np.maximum(goc_5yr[term_start_num] + m["fixed_spread_over_goc_5yr"] - mortgage_score_adj, m["fixed_rate_floor"])
new_rate = np.maximum(goc_5yr[renewal_rate_month] + m["fixed_spread_over_goc_5yr"] - mortgage_score_adj, m["fixed_rate_floor"])

# Simplification: we amortize at the current term's rate since signing. For a loan on its second
# term this slightly misstates the balance at renewal, but not the size of the rate jump.
months_paid = renewal_num - orig
old_payment = monthly_payment(loans["original_balance"], old_rate, amortization)
balance_at_renewal = balance_after(loans["original_balance"], old_rate, old_payment, months_paid)
new_payment = monthly_payment(balance_at_renewal, new_rate, amortization - months_paid)

loans["renews_in_window"] = renews
loans["renewal_month"] = pd.Series(calendar[renewal_num]).where(renews)
loans["old_rate"] = np.where(renews, old_rate, np.nan).round(2)
loans["new_rate"] = np.where(renews, new_rate, np.nan).round(2)
loans["balance_at_renewal"] = np.where(renews, balance_at_renewal, np.nan).round(2)
loans["old_payment"] = np.where(renews, old_payment, np.nan).round(2)
loans["new_payment"] = np.where(renews, new_payment, np.nan).round(2)
loans["payment_increase"] = (loans["new_payment"] - loans["old_payment"]).round(2)
loans["payment_shock_pct"] = (100 * (loans["new_payment"] / loans["old_payment"] - 1)).round(2)
loans["new_rate_source"] = np.where(
    ~renews, None,
    np.where(renewal_num <= last_month_num, "real rate", "assumed: last real rate held flat"),
)

log.info(f"{renews.sum():,} fixed mortgages renew between {window['renewal_window_start']} and {window['renewal_window_end']}")
loans[renews].head()

2026-10-17 19:01:59,696 | 01_simulate_loans | INFO | 6,716 fixed mortgages renew between 2025-01-01 and 2027-12-31


,loan_id,product,province,credit_score,rate_type,original_balance,orig_month_num,origination_month,rate_at_signing,renews_in_window,renewal_month,old_rate,new_rate,balance_at_renewal,old_payment,new_payment,payment_increase,payment_shock_pct,new_rate_source
27,28,mortgage,BC,697,fixed,797873.87,61,2020-02-01,2.71,True,2025-02-01,2.71,4.15,678283.21,3662.55,4164.18,501.63,13.70,real rate
36,37,mortgage,NB,668,fixed,337178.71,16,2016-05-01,2.19,True,2026-05-01,2.37,4.64,225584.78,1489.99,1742.18,252.19,16.93,real rate
73,74,mortgage,ON,738,fixed,817092.45,32,2017-09-01,3.09,True,2027-09-01,4.71,4.91,598065.17,4639.61,4701.78,62.17,1.34,assumed: last real rate held flat
75,76,mortgage,BC,821,fixed,569437.88,63,2020-04-01,1.72,True,2025-04-01,1.72,3.93,474253.66,2337.09,2857.35,520.26,22.26,real rate
85,86,mortgage,ON,678,fixed,449950.67,3,2015-04-01,2.27,True,2025-04-01,1.94,4.15,295541.07,1893.12,2208.10,314.98,16.64,real rate


## 6. Save

In [7]:
loans["source"] = "SIMULATED - not real customers"
loans = loans.drop(columns=["orig_month_num"])
loans.to_csv("data/loans.csv", index=False)
log.info(f"Saved data/loans.csv: {len(loans):,} rows, {len(loans.columns)} columns")

2026-10-17 19:02:00,522 | 01_simulate_loans | INFO | Saved data/loans.csv: 50,000 rows, 19 columns


## 7. Sanity checks — look at these, don't just trust them

In [8]:
print("Share of loans by product (config says 35% / 10% / 20% / 35%):")
print(loans["product"].value_counts(normalize=True).round(3))

renewing = loans[loans["renews_in_window"]].copy()
renewing["term_start_year"] = renewing["renewal_month"].dt.year - term // 12

print("\nRenewing fixed mortgages, by the year the current term started:")
renewing.groupby("term_start_year").agg(
    loans=("loan_id", "count"),
    median_old_rate=("old_rate", "median"),
    median_new_rate=("new_rate", "median"),
    median_shock_pct=("payment_shock_pct", "median"),
).round(2)

Share of loans by product (config says 35% / 10% / 20% / 35%):
product
card        0.352
mortgage    0.349
auto        0.200
heloc       0.098
Name: proportion, dtype: float64

Renewing fixed mortgages, by the year the current term started:


,loans,median_old_rate,median_new_rate,median_shock_pct
term_start_year,,,,
2020,2189,1.82,4.20,19.08
2021,2250,2.28,4.56,20.21
2022,2277,4.38,4.94,4.22


**Reading the table:** terms that started in 2020–21 locked in ~1.8–2.3% and renew at ~4.2–4.6%,
roughly a 20% higher payment. Terms that started in 2022 locked in after rates rose, so their
shock is small. With rates held flat, **no renewal in 2025–27 gets cheaper**: rates peaked in
late 2022 at about today's level. Loans that *will* get cheaper are the 2023–24 signings,
which renew in 2028–29 — outside this window.

In [9]:
cheaper = (renewing["payment_increase"] < 0).mean()
print(f"Share of 2025-27 renewals where the payment goes DOWN: {cheaper:.1%}")

Share of 2025-27 renewals where the payment goes DOWN: 0.0%
